# 08 — RQ1 Primary Inference: Median log Token Premium

**Protocol**: `NB08_RQ1_PROTOCOL_v001` · **Decision**: `RD-RQ1-FIRST-RESULT-01`

| record | SHA |
|---|---|
| base main | `79490b723ff4413763a84d310e50fa2748ccca6c` |
| decision | `e72274086a7e9c611c9014e6b5612df0e69dae30` |
| cohort | `9b695307c0551be84d4d6c374646bfe001b7b3a9` |
| protocol | `86521fdf04839d2e3e8e5db8e15a08ea067871e3` |

All three were committed **before** any result was observed.

**RQ1** — Under the fixed `o200k_base` Track A measurement and the final KO–EN semantically matched
cohort, is `Median(log Tokenization Premium) > 0`?

Every quantity below is computed from the D-04 physical artifact. No value is copied from EDA V1 or
EDA V2; EDA V2 is not an inference source. No raw KO/EN text is loaded or emitted.

Out of scope for this notebook: NB06, regex chunking, D-05, morphology explanatory models, M0–M3,
VIF/condition number, source_domain modeling, near-duplicate clustering, train/test split, NB10,
G-ID, causal claims.

## 01 — Canonical D-04 fail-closed validation

In [1]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import platform
import time
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb
import numpy as np
import pyarrow
import pyarrow.parquet as pq
import scipy
import scipy.stats as st

KST = ZoneInfo("Asia/Seoul")
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

D04 = ROOT / "data/registry/TOKEN_O200K_BASE_v001.parquet"
D04_SHA256 = "1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7"
PAIR_SET_HASH = "d9660d654ee449e4d0c23a0070225274"
EXPECTED_N = 3_835_988

BASE_MAIN_SHA = "79490b723ff4413763a84d310e50fa2748ccca6c"
RQ1_DECISION_SHA = "e72274086a7e9c611c9014e6b5612df0e69dae30"
RQ1_COHORT_SHA = "9b695307c0551be84d4d6c374646bfe001b7b3a9"
RQ1_PROTOCOL_SHA = "86521fdf04839d2e3e8e5db8e15a08ea067871e3"

BOOTSTRAP_B = 2000
BOOTSTRAP_SEED = 969634713
BOOTSTRAP_SEED_SOURCE = "RD-RQ1-FIRST-RESULT-01|NB08_RQ1_PROTOCOL_v001"
CI_Q = (0.025, 0.975)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()


assert hashlib.sha256(BOOTSTRAP_SEED_SOURCE.encode()).hexdigest()[:8] == f"{BOOTSTRAP_SEED:08x}", (
    "seed does not reproduce from its declared source string"
)

actual_sha = sha256_file(D04)
if actual_sha != D04_SHA256:
    raise SystemExit(f"CANONICAL_ARTIFACT_IDENTITY_MISMATCH: {actual_sha}")

schema = pq.read_schema(D04)
meta = pq.ParquetFile(D04).metadata
print(f"D-04 sha256    {actual_sha}  MATCH")
print(f"rows           {meta.num_rows:,}")
print(f"columns        {len(schema.names)}")
print(f"row groups     {meta.num_row_groups}")

D-04 sha256    1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7  MATCH
rows           3,835,988
columns        28
row groups     1535


## 02 — RQ1 cohort manifest validation

In [2]:
COHORT_MANIFEST = json.loads((ROOT / "ssot_nb01/02_ANALYSIS_COHORT_RQ1_v001.json").read_text())

con = duckdb.connect()
con.execute("SET memory_limit='5GB'")
con.execute("SET threads=8")
con.execute("SET preserve_insertion_order=false")
REL = f"read_parquet('{D04.as_posix()}')"

n_rows, n_distinct, n_null_id, n_null_y, n_nonfinite = con.execute(
    f"""SELECT count(*), count(DISTINCT pair_id),
              sum((pair_id IS NULL)::INT),
              sum((log_token_premium IS NULL)::INT),
              sum((NOT isfinite(log_token_premium))::INT)
       FROM {REL}"""
).fetchone()
pair_set_hash = con.execute(
    f"SELECT md5(string_agg(pair_id,'' ORDER BY pair_id)) FROM {REL}"
).fetchone()[0]

checks = {
    "row_count": (n_rows, EXPECTED_N),
    "distinct_pair_id": (n_distinct, EXPECTED_N),
    "null_pair_id": (n_null_id, 0),
    "null_outcome": (n_null_y, 0),
    "nonfinite_outcome": (n_nonfinite, 0),
    "pair_set_hash": (pair_set_hash, PAIR_SET_HASH),
    "manifest_row_count": (COHORT_MANIFEST["row_count"], EXPECTED_N),
    "manifest_status": (COHORT_MANIFEST["validation_status"], "PASS"),
}
for name, (got, want) in checks.items():
    status = "OK" if got == want else "FAIL"
    print(f"  {name:22s} {str(got):36s} {status}")
if any(got != want for got, want in checks.values()):
    raise SystemExit("RQ1_COHORT_VALIDATION_FAILED")

# Protocol section 7: hard fail on non-finite. No post-hoc deletion is permitted.
if n_nonfinite != 0:
    raise SystemExit("NONFINITE_OUTCOME_HARD_FAIL")
print("\nRQ1_COHORT_VALIDATED")

  row_count              3835988                              OK
  distinct_pair_id       3835988                              OK
  null_pair_id           0                                    OK
  null_outcome           0                                    OK
  nonfinite_outcome      0                                    OK
  pair_set_hash          d9660d654ee449e4d0c23a0070225274     OK
  manifest_row_count     3835988                              OK
  manifest_status        PASS                                 OK

RQ1_COHORT_VALIDATED


## 03 — Descriptive outcome snapshot

The outcome column is materialised once. Only `log_token_premium` and the direction stratum are
loaded — no token ID arrays and no text.

In [3]:
t0 = time.monotonic()
tbl = con.execute(
    f"""SELECT t.log_token_premium AS y,
              (p.translation_direction IS DISTINCT FROM 'UNKNOWN') AS known_dir
       FROM {REL} t
       JOIN read_parquet('{(ROOT / 'data/registry/PAIR_REGISTRY_v002.parquet').as_posix()}') p
         USING (pair_id)"""
).fetch_arrow_table()
Y = tbl.column("y").to_numpy(zero_copy_only=False).astype(np.float64)
KNOWN = tbl.column("known_dir").to_numpy(zero_copy_only=False).astype(bool)
del tbl
load_sec = round(time.monotonic() - t0, 2)

assert Y.shape[0] == EXPECTED_N, Y.shape
assert np.isfinite(Y).all(), "NONFINITE_OUTCOME_HARD_FAIL"

n_known = int(KNOWN.sum())
print(f"loaded {Y.size:,} outcomes in {load_sec}s")
print(f"known-direction rows {n_known:,}   unknown {Y.size - n_known:,}")

desc = {
    "n": int(Y.size),
    "mean": float(Y.mean()),
    "sd": float(Y.std(ddof=1)),
    "min": float(Y.min()),
    "p01": float(np.percentile(Y, 1)),
    "p25": float(np.percentile(Y, 25)),
    "median": float(np.median(Y)),
    "p75": float(np.percentile(Y, 75)),
    "p99": float(np.percentile(Y, 99)),
    "max": float(Y.max()),
    "n_zero_exact": int((Y == 0.0).sum()),
    "n_positive": int((Y > 0.0).sum()),
    "n_negative": int((Y < 0.0).sum()),
}
desc["share_TP_gt_1"] = desc["n_positive"] / desc["n"]
for k, v in desc.items():
    print(f"  {k:16s} {v}")

/tmp/ipykernel_209655/2377813471.py:8: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  ).fetch_arrow_table()


loaded 3,835,988 outcomes in 0.84s
known-direction rows 3,785,441   unknown 50,547
  n                3835988
  mean             0.28517678624044906
  sd               0.22209552728201393
  min              -2.74859629549656
  p01              -0.2876820724517809
  p25              0.15415067982725836
  median           0.28768207245178085
  p75              0.42744401482693967
  p99              0.8109302162163288
  max              3.6375861597263857
  n_zero_exact     196718
  n_positive       3375095
  n_negative       264175
  share_TP_gt_1    0.8798502497922308


## 04 — Wilcoxon signed-rank (primary test)

Frozen settings: `alternative='greater'`, `zero_method='wilcox'` (exact zeros dropped).
The p-value is never reported as 0; on underflow `log10(p)` is derived from the normal
approximation.

In [4]:
def log10_p_from_normal(z: float) -> float:
    """log10 upper-tail probability of the standard normal, underflow-safe."""
    return float(st.norm.logsf(z) / np.log(10.0))


def wilcoxon_report(y: np.ndarray, label: str) -> dict:
    n_zero = int((y == 0.0).sum())
    res = st.wilcoxon(y, alternative="greater", zero_method="wilcox")
    nz = y[y != 0.0]
    n_eff = nz.size
    # Normal-approximation z for underflow-safe log10(p), matching zero_method='wilcox'.
    ranks = st.rankdata(np.abs(nz))
    w_plus = float(ranks[nz > 0].sum())
    mu = n_eff * (n_eff + 1) / 4.0
    # tie correction on absolute-value ranks
    _, counts = np.unique(np.abs(nz), return_counts=True)
    tie_term = float(((counts ** 3 - counts).sum()) / 48.0)
    sigma = float(np.sqrt(n_eff * (n_eff + 1) * (2 * n_eff + 1) / 24.0 - tie_term))
    z = (w_plus - mu) / sigma
    log10p = log10_p_from_normal(z)
    out = {
        "label": label,
        "n_total": int(y.size),
        "n_zero_dropped": n_zero,
        "n_effective": int(n_eff),
        "statistic": float(res.statistic),
        "w_plus": w_plus,
        "z_normal_approx": float(z),
        "pvalue_raw": float(res.pvalue),
        "pvalue_underflowed": bool(res.pvalue == 0.0),
        "log10_pvalue": log10p,
        "alternative": "greater",
        "zero_method": "wilcox",
    }
    disp = (
        f"p < 1e-300 (underflow; log10(p) = {log10p:.1f})"
        if out["pvalue_underflowed"]
        else f"p = {out['pvalue_raw']:.6g}"
    )
    out["pvalue_reported"] = disp
    print(f"[{label}] W={out['statistic']:.6g}  zeros dropped={n_zero}  n_eff={n_eff:,}")
    print(f"[{label}] z={z:.3f}  {disp}")
    return out


t0 = time.monotonic()
wil_primary = wilcoxon_report(Y, "PRIMARY_FINAL_COHORT")
wil_sec = round(time.monotonic() - t0, 2)
print(f"\nwilcoxon runtime {wil_sec}s")

[PRIMARY_FINAL_COHORT] W=6.40555e+12  zeros dropped=196718  n_eff=3,639,270
[PRIMARY_FINAL_COHORT] z=1544.070  p < 1e-300 (underflow; log10(p) = -517715.4)

wilcoxon runtime 0.64s


## 05 — Sign test (mandatory robustness)

Ties (`Y == 0`) are excluded from the binomial denominator. The tie count and share are reported
separately, as the protocol requires.

In [5]:
def sign_report(y: np.ndarray, label: str) -> dict:
    pos = int((y > 0.0).sum())
    neg = int((y < 0.0).sum())
    ties = int((y == 0.0).sum())
    n_eff = pos + neg
    res = st.binomtest(pos, n=n_eff, p=0.5, alternative="greater")
    # underflow-safe log10(p) via the normal approximation to Binomial(n_eff, 0.5)
    z = (pos - n_eff / 2.0) / np.sqrt(n_eff / 4.0)
    log10p = log10_p_from_normal(z)
    out = {
        "label": label,
        "positive": pos,
        "negative": neg,
        "ties": ties,
        "tie_share": ties / y.size,
        "n_effective": n_eff,
        "pvalue_raw": float(res.pvalue),
        "pvalue_underflowed": bool(res.pvalue == 0.0),
        "log10_pvalue": log10p,
        "z_normal_approx": float(z),
        "alternative": "greater",
    }
    disp = (
        f"p < 1e-300 (underflow; log10(p) = {log10p:.1f})"
        if out["pvalue_underflowed"]
        else f"p = {out['pvalue_raw']:.6g}"
    )
    out["pvalue_reported"] = disp
    print(f"[{label}] +{pos:,}  -{neg:,}  ties={ties:,} ({out['tie_share']:.6%})  n_eff={n_eff:,}")
    print(f"[{label}] {disp}")
    return out


sign_primary = sign_report(Y, "PRIMARY_FINAL_COHORT")

[PRIMARY_FINAL_COHORT] +3,375,095  -264,175  ties=196,718 (5.128223%)  n_eff=3,639,270
[PRIMARY_FINAL_COHORT] p < 1e-300 (underflow; log10(p) = -577458.1)


## 06 — Bootstrap equivalence benchmark

The full run uses a monotone-indexing shortcut: because the outcome is sorted once, the median of a
resample equals the sorted array evaluated at the middle order statistics of the drawn indices. This
cell verifies that shortcut against a direct `np.median` reference on the same drawn indices. Both
must agree bit-for-bit; any mismatch invalidates the CI regardless of runtime.

In [6]:
Y_SORTED = np.sort(Y)
N = Y_SORTED.size
LO, HI = (N // 2) - 1, N // 2  # N is even; median averages these two order statistics
assert N % 2 == 0, N


def boot_medians_fast(y_sorted: np.ndarray, b: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = y_sorted.size
    out = np.empty(b, dtype=np.float64)
    for i in range(b):
        idx = rng.integers(0, n, size=n)
        part = np.partition(idx, (LO, HI))
        out[i] = 0.5 * (y_sorted[part[LO]] + y_sorted[part[HI]])
    return out


def boot_medians_reference(y_sorted: np.ndarray, b: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = y_sorted.size
    out = np.empty(b, dtype=np.float64)
    for i in range(b):
        idx = rng.integers(0, n, size=n)
        out[i] = float(np.median(y_sorted[idx]))
    return out


BENCH_B = 25
t0 = time.monotonic()
bench_fast = boot_medians_fast(Y_SORTED, BENCH_B, BOOTSTRAP_SEED)
bench_ref = boot_medians_reference(Y_SORTED, BENCH_B, BOOTSTRAP_SEED)
bench_sec = round(time.monotonic() - t0, 2)

mismatch = int((bench_fast != bench_ref).sum())
equivalence = {
    "benchmark_replicates": BENCH_B,
    "seed": BOOTSTRAP_SEED,
    "mismatch_replicates": mismatch,
    "max_abs_difference": float(np.max(np.abs(bench_fast - bench_ref))),
    "status": "PASS" if mismatch == 0 else "FAIL",
    "runtime_sec": bench_sec,
    "note": "monotone-indexing shortcut vs direct np.median on identical drawn indices",
}
print(f"replicates={BENCH_B}  mismatch={mismatch}  max|diff|={equivalence['max_abs_difference']}")
print(f"BOOTSTRAP_EQUIVALENCE_STATUS = {equivalence['status']}")
if mismatch != 0:
    raise SystemExit("BOOTSTRAP_EQUIVALENCE_FAILED")

replicates=25  mismatch=0  max|diff|=0.0
BOOTSTRAP_EQUIVALENCE_STATUS = PASS


## 07 — Full primary bootstrap CI

`B = 2000`, pair-level iid resampling, seed `969634713`, percentile quantiles 0.025 / 0.975.

In [7]:
t0 = time.monotonic()
boot_primary = boot_medians_fast(Y_SORTED, BOOTSTRAP_B, BOOTSTRAP_SEED)
boot_sec = round(time.monotonic() - t0, 2)

median_primary = float(np.median(Y))
ci_primary = [float(np.quantile(boot_primary, CI_Q[0])), float(np.quantile(boot_primary, CI_Q[1]))]
exp_median_primary = float(np.exp(median_primary))

print(f"B={BOOTSTRAP_B}  seed={BOOTSTRAP_SEED}  runtime {boot_sec}s")
print(f"median(logTP)      {median_primary:.10f}")
print(f"95% percentile CI  [{ci_primary[0]:.10f}, {ci_primary[1]:.10f}]")
print(f"exp(median)        {exp_median_primary:.10f}")
print(f"bootstrap replicate sd {float(boot_primary.std(ddof=1)):.3e}")

B=2000  seed=969634713  runtime 58.1s
median(logTP)      0.2876820725
95% percentile CI  [0.2876820725, 0.2876820725]
exp(median)        1.3333333333
bootstrap replicate sd 5.553e-17


## 08 — Known-direction sensitivity

`translation_direction != 'UNKNOWN'`. Same estimand, tests, bootstrap settings and seed. This is
reported beside the primary result and does not replace it.

In [8]:
Y_KNOWN = Y[KNOWN]
print(f"known-direction N = {Y_KNOWN.size:,}")

wil_known = wilcoxon_report(Y_KNOWN, "KNOWN_DIRECTION_ONLY")
sign_known = sign_report(Y_KNOWN, "KNOWN_DIRECTION_ONLY")

Y_KNOWN_SORTED = np.sort(Y_KNOWN)
NK = Y_KNOWN_SORTED.size
LOK, HIK = (NK - 1) // 2, NK // 2  # handles odd or even NK


def boot_medians_fast_k(y_sorted: np.ndarray, b: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = y_sorted.size
    out = np.empty(b, dtype=np.float64)
    for i in range(b):
        idx = rng.integers(0, n, size=n)
        part = np.partition(idx, (LOK, HIK))
        out[i] = 0.5 * (y_sorted[part[LOK]] + y_sorted[part[HIK]])
    return out


t0 = time.monotonic()
boot_known = boot_medians_fast_k(Y_KNOWN_SORTED, BOOTSTRAP_B, BOOTSTRAP_SEED)
boot_known_sec = round(time.monotonic() - t0, 2)

median_known = float(np.median(Y_KNOWN))
ci_known = [float(np.quantile(boot_known, CI_Q[0])), float(np.quantile(boot_known, CI_Q[1]))]
exp_median_known = float(np.exp(median_known))
share_gt1_known = float((Y_KNOWN > 0).sum() / Y_KNOWN.size)

print(f"median(logTP)      {median_known:.10f}")
print(f"95% percentile CI  [{ci_known[0]:.10f}, {ci_known[1]:.10f}]")
print(f"exp(median)        {exp_median_known:.10f}")

known-direction N = 3,785,441


[KNOWN_DIRECTION_ONLY] W=6.23753e+12  zeros dropped=194084  n_eff=3,591,357
[KNOWN_DIRECTION_ONLY] z=1533.636  p < 1e-300 (underflow; log10(p) = -510742.4)
[KNOWN_DIRECTION_ONLY] +3,330,539  -260,818  ties=194,084 (5.127117%)  n_eff=3,591,357
[KNOWN_DIRECTION_ONLY] p < 1e-300 (underflow; log10(p) = -569765.7)


median(logTP)      0.2876820725
95% percentile CI  [0.2876820725, 0.2876820725]
exp(median)        1.3333333333


## 09 — Primary results table

In [9]:
rows = [
    {
        "cohort": "PRIMARY_FINAL_COHORT",
        "N": int(Y.size),
        "median_logTP": median_primary,
        "bootstrap_95CI_low": ci_primary[0],
        "bootstrap_95CI_high": ci_primary[1],
        "exp_median_logTP": exp_median_primary,
        "wilcoxon_W": wil_primary["statistic"],
        "wilcoxon_p": wil_primary["pvalue_reported"],
        "sign_positive": sign_primary["positive"],
        "sign_negative": sign_primary["negative"],
        "sign_ties": sign_primary["ties"],
        "sign_p": sign_primary["pvalue_reported"],
        "share_TP_gt_1": desc["share_TP_gt_1"],
    },
    {
        "cohort": "KNOWN_DIRECTION_ONLY",
        "N": int(Y_KNOWN.size),
        "median_logTP": median_known,
        "bootstrap_95CI_low": ci_known[0],
        "bootstrap_95CI_high": ci_known[1],
        "exp_median_logTP": exp_median_known,
        "wilcoxon_W": wil_known["statistic"],
        "wilcoxon_p": wil_known["pvalue_reported"],
        "sign_positive": sign_known["positive"],
        "sign_negative": sign_known["negative"],
        "sign_ties": sign_known["ties"],
        "sign_p": sign_known["pvalue_reported"],
        "share_TP_gt_1": share_gt1_known,
    },
]

hdr = ["cohort", "N", "median_logTP", "bootstrap_95CI_low", "bootstrap_95CI_high",
       "exp_median_logTP", "wilcoxon_W", "wilcoxon_p", "sign_positive", "sign_negative",
       "sign_ties", "sign_p", "share_TP_gt_1"]
print("| " + " | ".join(hdr) + " |")
print("|" + "|".join(["---"] * len(hdr)) + "|")
for r in rows:
    cells = []
    for h in hdr:
        v = r[h]
        cells.append(f"{v:,}" if isinstance(v, int) else (f"{v:.10g}" if isinstance(v, float) else str(v)))
    print("| " + " | ".join(cells) + " |")

print("\nshare_TP_gt_1 is a descriptive contextual statistic. It is NOT the sign test and does")
print("not carry the sign test's inferential role.")

| cohort | N | median_logTP | bootstrap_95CI_low | bootstrap_95CI_high | exp_median_logTP | wilcoxon_W | wilcoxon_p | sign_positive | sign_negative | sign_ties | sign_p | share_TP_gt_1 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| PRIMARY_FINAL_COHORT | 3,835,988 | 0.2876820725 | 0.2876820725 | 0.2876820725 | 1.333333333 | 6.405551963e+12 | p < 1e-300 (underflow; log10(p) = -517715.4) | 3,375,095 | 264,175 | 196,718 | p < 1e-300 (underflow; log10(p) = -577458.1) | 0.8798502498 |
| KNOWN_DIRECTION_ONLY | 3,785,441 | 0.2876820725 | 0.2876820725 | 0.2876820725 | 1.333333333 | 6.237534312e+12 | p < 1e-300 (underflow; log10(p) = -510742.4) | 3,330,539 | 260,818 | 194,084 | p < 1e-300 (underflow; log10(p) = -569765.7) | 0.8798285325 |

share_TP_gt_1 is a descriptive contextual statistic. It is NOT the sign test and does
not carry the sign test's inferential role.


## 10 — Result artifact and claim boundary

In [10]:
results = {
    "artifact_id": "NB08_RQ1_RESULTS_v001",
    "decision_id": "RD-RQ1-FIRST-RESULT-01",
    "protocol_id": "NB08_RQ1_PROTOCOL_v001",
    "change_requests": ["CR-RQ1-BOOTSTRAP-FAST-2000-01"],
    "git": {
        "base_main_sha": BASE_MAIN_SHA,
        "rq1_decision_sha": RQ1_DECISION_SHA,
        "rq1_cohort_sha": RQ1_COHORT_SHA,
        "rq1_protocol_sha": RQ1_PROTOCOL_SHA,
        "branch": "research/nb08-rq1-primary-20260817",
    },
    "source": {
        "artifact_path": str(D04.relative_to(ROOT)),
        "artifact_sha256": actual_sha,
        "pair_set_hash": pair_set_hash,
        "row_count": int(n_rows),
        "distinct_pair_id": int(n_distinct),
        "eda_v2_used_as_inference_source": False,
    },
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "duckdb": duckdb.__version__,
        "pyarrow": pyarrow.__version__,
    },
    "outcome": "log_token_premium",
    "estimand": "Median(log_token_premium)",
    "hypothesis": {"H0": "theta = 0", "H1": "theta > 0", "alternative": "greater"},
    "descriptive": desc,
    "primary": {
        "cohort": "PRIMARY_FINAL_COHORT",
        "n": int(Y.size),
        "median_logTP": median_primary,
        "exp_median_logTP": exp_median_primary,
        "bootstrap_ci95": ci_primary,
        "wilcoxon": wil_primary,
        "sign_test": sign_primary,
        "share_TP_gt_1": desc["share_TP_gt_1"],
    },
    "sensitivity_known_direction": {
        "cohort": "KNOWN_DIRECTION_ONLY",
        "n": int(Y_KNOWN.size),
        "median_logTP": median_known,
        "exp_median_logTP": exp_median_known,
        "bootstrap_ci95": ci_known,
        "wilcoxon": wil_known,
        "sign_test": sign_known,
        "share_TP_gt_1": share_gt1_known,
    },
    "bootstrap": {
        "method": "percentile, pair_id row-level iid resampling",
        "implementation": "monotone-indexing order-statistic shortcut on the sorted outcome",
        "B": BOOTSTRAP_B,
        "seed": BOOTSTRAP_SEED,
        "seed_source_string": BOOTSTRAP_SEED_SOURCE,
        "seed_sha256": hashlib.sha256(BOOTSTRAP_SEED_SOURCE.encode()).hexdigest(),
        "quantiles": list(CI_Q),
        "equivalence_audit": equivalence,
        "runtime_sec_primary": boot_sec,
        "runtime_sec_known_direction": boot_known_sec,
        "resampling_unit_caveat": (
            "Row-level iid bootstrap does not account for source dependence or source imbalance. "
            "Source-stratified and dependence-sensitive robustness is deferred to an NB08 appendix "
            "or NB11 and is not a prerequisite for this first-result release."
        ),
    },
    "runtime_sec": {"load": load_sec, "wilcoxon": wil_sec},
    "policies": {
        "zero_method": "wilcox",
        "sign_test_ties": "excluded from the binomial denominator",
        "nonfinite_policy": "hard fail, no post-hoc deletion",
        "pvalue_policy": "p = 0 is never reported; log10(p) given on underflow",
        "primary_exclusion_rule": "NONE beyond the frozen final cohort",
    },
    "claim_status": {
        "permitted": (
            "Under the fixed o200k_base raw-text Track A measurement and the defined final paired "
            "KO-EN cohort, statistical evidence was observed that the pair-level median log token "
            "premium is greater than zero."
        ),
        "prohibited": [
            "Korean is intrinsically inefficient for AI",
            "generalization to all tokenizers",
            "morphology is the cause",
            "any domain effect claim",
            "reasoning degradation",
            "any fixed API cost increase figure claimed unconditionally",
            "any causal language",
        ],
        "notes": [
            "exp(median(logTP)) is a median-scale quantity and is NOT the aggregate token ratio.",
            "share_TP_gt_1 is descriptive context, not the sign test.",
            "Wilcoxon signed-rank carries a symmetry/location-shift interpretation; the sign test "
            "is the distribution-free robustness check on the median estimand.",
        ],
    },
    "created_at_kst": dt.datetime.now(KST).isoformat(timespec="seconds"),
}

out_path = ROOT / "ssot_nb01/04_NB08_RQ1_RESULTS_v001.json"
out_path.write_text(json.dumps(results, ensure_ascii=False, indent=2, sort_keys=True) + "\n")
con.close()
print(f"written {out_path.relative_to(ROOT)}")
print("\nRQ1_PRIMARY_INFERENCE_COMPLETE")

written ssot_nb01/04_NB08_RQ1_RESULTS_v001.json

RQ1_PRIMARY_INFERENCE_COMPLETE
